# mmWave Radar Sensing: Analytic Point Targets

This notebook is a lightweight sanity check for FMCW point-target synthesis. It intentionally bypasses Sionna ray tracing, scene meshes, and PO/RT coupling so the range and virtual-array angle responses can be inspected in isolation.

The examples use the generic helpers in `mmWaveRadar.simulation.point_targets`. The model is a far-field virtual-center approximation: each target contributes one round-trip delay shared by all virtual channels, while the virtual-channel phase comes from the array coordinate and target direction. This is useful for range/angle debugging and board-layout comparisons; it is not a near-field per-channel path-length model.


In [ ]:
from pathlib import Path
import os
import sys
import tempfile

# Let the notebook run from the repository root or from demo/ without requiring
# an editable install. The loop stops at the first parent containing src/mmWaveRadar.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src" / "mmWaveRadar").exists():
    repo_root = repo_root.parent
if not (repo_root / "src" / "mmWaveRadar").exists():
    raise RuntimeError("Run this notebook from inside a HERMES source checkout, including the top-level demo/ directory.")
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Keep Matplotlib font/cache files out of the repository when the notebook is
# run in a fresh environment.
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "mmwave-radar-mpl"))

import matplotlib.pyplot as plt
import numpy as np

from mmWaveRadar.dsp import plot_axis_image, plot_range_profile, range_fft
from mmWaveRadar.radar import RadarHardware, available_ti_boards, get_ti_board_spec
from mmWaveRadar.simulation import (
    azimuth_elevation_axes,
    db_relative,
    make_point_target_fmcw_config,
    range_resolved_angle_fft,
    synthesize_virtual_center_point_targets,
)


In [ ]:
# Use an approximate generic xWR68xx-style layout for the first range-profile
# check. This path does not depend on documented board virtual-channel metadata;
# the simulator helper derives virtual coordinates from Tx/Rx geometry.
hardware = RadarHardware.from_xwr68xx("IWR6843AOP")

# The FMCW profile uses the notebook/demo defaults unless overridden. Tie num_tx
# to the hardware so chirp/frame metadata stays consistent with the array.
fmcw = make_point_target_fmcw_config(
    carrier_frequency=60e9,
    num_tx=hardware.num_tx,
)
print(f"Virtual channels: {hardware.num_virtual_channels}")


In [ ]:
# Targets are specified in range plus azimuth/elevation relative to radar
# boresight. Reflectivity is a complex voltage coefficient before optional
# range spreading and channel gains.
range_profile_targets = [
    {"range_m": 1.35, "azimuth_deg": -8.5, "elevation_deg": 3.8, "reflectivity": 1.0 + 0.0j},
    {"range_m": 2.10, "azimuth_deg": 15.0, "elevation_deg": -6.5, "reflectivity": 0.45 * np.exp(1j * 0.4)},
]

# For this simple range-profile view, disable fixed channel phase, antenna gain,
# and patterns so the plot mainly shows the beat frequencies from target ranges.
range_profile_result = synthesize_virtual_center_point_targets(
    hardware,
    fmcw,
    range_profile_targets,
    include_channel_phase=False,
    include_antenna_gain=False,
    include_pattern=False,
)
adc = range_profile_result.adc
print(adc.shape)  # [num_adc_samples, num_virtual_channels]


In [ ]:
# Compute the positive-frequency range FFT for one chirp. Zero-padding via
# nfft_mult makes the plotted peak locations easier to read; it does not change
# the physical range resolution set by bandwidth.
rt, ranges_m = range_fft(adc[None, :, :], fmcw=fmcw, window="Hann", nfft_mult=8)
profile = rt[0, :, 0]

fig, ax = plt.subplots(figsize=(7, 3), constrained_layout=True)
plot_range_profile(
    ranges_m,
    20 * np.log10(np.abs(profile) + 1e-12),
    ax=ax,
    range_limits_m=(0.0, 3.0),
    xlabel="Range (m)",
    ylabel="Magnitude (dB)",
    title="Point-target range profile, first virtual channel, Hann window",
)
plt.show()


## Documented Board Virtual-Array Point Targets

This section compares documented board virtual arrays using the same analytic target scene. The board catalog only supplies hardware metadata: design carrier frequency, documented virtual-channel positions, fixed phase signs, and optional digitized antenna patterns. Synthesis and angle processing still use the generic point-target helpers.

Both target cases place targets at the same range so the angle maps test virtual-array angular response rather than range separability.


In [ ]:
# Example board layouts with different virtual-array apertures. The keys are
# resolved through the local board catalog, but the downstream synthesis call is
# still generic over RadarHardware + FMCWConfig.
ti_board_keys = ["AWRL6844EVM", "IWR6843ISK", "IWR6843AOPEVM"]
assert set(ti_board_keys).issubset(set(available_ti_boards()))

ti_shared_target_range_m = 1.60
ti_target_cases = [
    {
        "name": "Separated angles",
        "range_m": ti_shared_target_range_m,
        "targets": [
            {"range_m": ti_shared_target_range_m, "azimuth_deg": -24.0, "elevation_deg": -10.0, "reflectivity": 1.0 + 0.0j},
            {"range_m": ti_shared_target_range_m, "azimuth_deg": 28.0, "elevation_deg": 12.0, "reflectivity": 0.90 * np.exp(1j * 0.4)},
        ],
    },
    {
        "name": "Close azimuth angles",
        "range_m": ti_shared_target_range_m,
        "targets": [
            {"range_m": ti_shared_target_range_m, "azimuth_deg": -5.0, "elevation_deg": 0.0, "reflectivity": 1.0 + 0.0j},
            {"range_m": ti_shared_target_range_m, "azimuth_deg": 5.0, "elevation_deg": 0.0, "reflectivity": 0.90 * np.exp(1j * 0.4)},
        ],
    },
]

# Pattern curves are off by default to isolate virtual-array geometry. Fixed
# channel signs are applied during synthesis and then removed before angle FFT;
# this exercises the calibration path without changing the expected peak angles.
ti_include_digitized_patterns = False
ti_apply_fixed_channel_signs = True
ti_calibrate_fixed_channel_signs_before_fft = True


In [ ]:
ti_case_results = []
pattern_mode = "digitized" if ti_include_digitized_patterns else "none"

for case in ti_target_cases:
    board_results = []
    angle_results = []
    for board_key in ti_board_keys:
        # Board-specific work stops at constructing hardware and picking the
        # carrier frequency. The point-target synthesis below is board-agnostic.
        spec = get_ti_board_spec(board_key)
        hardware = RadarHardware.from_ti_board(board_key, pattern_mode=pattern_mode)
        fmcw = make_point_target_fmcw_config(
            carrier_frequency=spec.design_frequency_hz,
            num_tx=hardware.num_tx,
        )
        result = synthesize_virtual_center_point_targets(
            hardware,
            fmcw,
            case["targets"],
            include_channel_phase=ti_apply_fixed_channel_signs,
            include_antenna_gain=False,
            include_pattern=ti_include_digitized_patterns,
        )
        board_results.append(result)

        # Convert ADC -> range bins -> virtual-array grid -> 2D angle FFT. If
        # fixed channel signs were included above, remove them before the FFT.
        angle_results.append(
            range_resolved_angle_fft(
                result.adc,
                hardware,
                fmcw,
                calibrate_channel_phase=ti_calibrate_fixed_channel_signs_before_fft,
            )
        )
    ti_case_results.append({**case, "board_results": board_results, "angle_results": angle_results})

fig, axes = plt.subplots(len(ti_case_results), len(ti_board_keys), figsize=(15, 8.6), constrained_layout=True, sharex=True, sharey=True)
axes = np.asarray(axes)
if axes.ndim == 1:
    axes = axes[None, :]

for row, case_result in enumerate(ti_case_results):
    for col, (result, angle_result) in enumerate(zip(case_result["board_results"], case_result["angle_results"])):
        ax = axes[row, col]

        # Plot the angle map at the range bin nearest the shared target range.
        # FFT axes are direction cosines (u, v), converted to degrees for display.
        range_index = int(np.argmin(np.abs(angle_result["ranges_m"] - case_result["range_m"])))
        power_2d = np.asarray(angle_result["angle"]["map"])[range_index]
        azimuth_deg, elevation_deg = azimuth_elevation_axes(angle_result["angle"])
        image_db = db_relative(power_2d)
        im = plot_axis_image(image_db, azimuth_deg, elevation_deg, ax=ax, cmap="magma", vmin=-35.0, vmax=0.0)
        if row == 0:
            ax.set_title(result.hardware.name)
        if col == 0:
            ax.set_ylabel(f"{case_result['name']}\nElevation angle (deg)")
        if row == len(ti_case_results) - 1:
            ax.set_xlabel("Azimuth angle (deg)")
        ax.set_xlim(-70.0, 70.0)
        ax.set_ylim(-45.0, 45.0)
        ax.text(
            0.02,
            0.98,
            f"range bin {angle_result['ranges_m'][range_index]:.2f} m",
            transform=ax.transAxes,
            va="top",
            fontsize=9,
            bbox={"facecolor": "white", "edgecolor": "0.8", "alpha": 0.85},
        )

        # Cyan crosses mark the requested target angles; they are a quick visual
        # check for angle-axis orientation and board aperture differences.
        for target in case_result["targets"]:
            ax.plot(target["azimuth_deg"], target["elevation_deg"], "cx", markersize=8, markeredgewidth=2)
fig.colorbar(im, ax=axes, label="Relative power (dB)")
plt.show()

for case_result in ti_case_results:
    print(f"Case: {case_result['name']} at {case_result['range_m']:.2f} m")
    for result in case_result["board_results"]:
        hardware = result.hardware
        fmcw = result.fmcw
        print(f"  {hardware.name}: {hardware.num_tx} TX x {hardware.num_rx} RX -> {hardware.num_virtual_channels} virtual, fc={fmcw.carrier_frequency / 1e9:.1f} GHz")
    print()
